In [2]:
from models.PP3DR_depth_focal import PP3DR
import torch
from datasets.nrgbd_dataset import nrgbd_dataset
import matplotlib.pyplot as plt

In [ ]:
model = PP3DR()
remove_front = False

loaded_state_dict = torch.load("/vulcanscratch/hughma/PP3DR/median-100/PP3DR.pth", weights_only=True)
if remove_front:
    # The lines here are necessary if all the loaded state dict entries begin with an extra "module."
    new_state_dict = {}
    for key in loaded_state_dict:
        new_state_dict[key[7:]] = loaded_state_dict[key]
    model.load_state_dict(new_state_dict)
else:
    model.load_state_dict(loaded_state_dict)
print("post load")

model = model.eval().to("cuda")
dataset = nrgbd_dataset()
data = dataset[0]
print("post dataset")
for key in data:
    data[key] = data[key].unsqueeze(0)
with torch.amp.autocast("cuda", dtype = torch.bfloat16), torch.no_grad():
    pred = model(data['images'].to("cuda"), data['rope_x'].to("cuda"), data['rope_y'].to("cuda"))
print("post pred")
for key in pred:
    pred[key] = pred[key].to(torch.float32).cpu().numpy(force=True)
    print(f"{key}: {pred[key].shape}")
print("done")

In [ ]:
# plt.style.use("seaborn-v0_8-darkgrid")
plt.style.use("classic")
fig, (a1, a2) = plt.subplots(1, 2)
a1.imshow(data['depths'][0][0])
a2.imshow(pred['log_depths'][0][0])
fig.tight_layout()